# Simulating the X-ray luminosity function (XLF) of an X-ray binary (XRB) population

#### Anastasios Fragos — POSYDON School 2026, Wednesday 26 August, 10:00–11:00

In the previous labs, you explored populations of double compact object
mergers, supernovae, and gamma-ray bursts. These phenomena are instantaneous
transient events: they occur on timescales much shorter than the typical
lifetime of a binary system. Within the POSYDON framework, such events can be
modeled efficiently using an initial–final interpolation scheme, which maps
the initial binary properties directly to their final, post-event outcome.

X-ray binaries, however, are fundamentally different. They represent
evolutionary phases of binary systems that persist for a non-negligible
fraction of the system's lifetime. During these phases, properties such as the
component masses, orbital period, mass-transfer rate, and X-ray luminosity
evolve continuously with time. Because of this temporal evolution, an
initial–final interpolation approach is not sufficient to describe everything
we may observe from an XRB.

### Modeling strategies for X-ray binary populations

There are two principal approaches to model populations of systems like X-ray
binaries:

1. **Snapshot method**
   - Evolve each binary up to a pre-specified age.
   - Record its properties at that moment.
   - If the binary is in an X-ray phase at that time, include it in the
     population statistics.
2. **Full evolutionary history method**
   - Evolve each binary from zero age to the end of its life.
   - Identify all intervals during which the system qualifies as an XRB.
   - Record the evolving properties across those intervals and weight them by
     the time spent in each state.

The snapshot method is conceptually straightforward but computationally
inefficient: a binary may have been an X-ray source at earlier or later times,
and its observable properties evolve during the X-ray phase, which is not
fully captured by a single snapshot. The full-history method uses the
simulation more efficiently and captures the temporal evolution in greater
detail, but it is conceptually more complex to implement.

![XRB schematic](xrb_schematic.png)

<details>
<summary>Code to produce this diagram</summary>

~~~python
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 3))
ax.hlines(1, 0, 11, color="black")
ax.set_ylim(0.5, 1.6)
ax.set_xlim(0, 11)

events = [0, 2, 8, 10]
labels = ["ZAMS", "1st Supernova", "2nd Supernova", "DCO formation"]
colors = ["black", "red", "red", "orange"]

for x, label, color in zip(events, labels, colors):
    ax.plot(x, 1, "o", color=color)
    ax.text(x, 1.12, label, ha="center", color=color, fontsize=10)

ax.hlines(0.9, 3, 7, color="blue", linewidth=6, alpha=0.3)
ax.text(5, 0.72, "X-ray Binary Phase", ha="center", color="blue")
ax.annotate("Snapshot method\n(one age)", xy=(6, 1.05), xytext=(6, 1.38),
            arrowprops=dict(arrowstyle="->", color="darkgreen"),
            ha="center", color="darkgreen", fontsize=9)
ax.annotate("", xy=(3, 1.28), xytext=(7, 1.28),
            arrowprops=dict(arrowstyle="|-|", color="purple", linewidth=2))
ax.text(5, 1.35, "Full history method", ha="center", color="purple",
        fontsize=9)
ax.axis("off")
ax.set_title("From ZAMS to Double Compact Object Formation", fontsize=14)
plt.savefig("xrb_schematic.png", dpi=150, bbox_inches="tight")
plt.show()
~~~

</details>

In this first lab, we adopt the snapshot method: we evolve binaries to a fixed
age and record their properties if they are XRBs at that time. This introduces
the concepts and workflow for simulating an XRB population. A full-history
treatment is discussed for context but is deferred to a later tutorial.

**By the end of this exercise, you should be able to:**

- set up and run a POSYDON population synthesis calculation appropriate for
  modeling high-mass X-ray binaries;
- load and inspect simulated populations, extracting the number of systems,
  simulated stellar mass, and formation pathways;
- identify XRBs at a fixed age using stellar states and accretion properties;
- map POSYDON's S1/S2 notation to donor/accretor quantities;
- calculate RLO, wind-fed, and Be-XRB luminosities using the new `xrb` module;
- construct and normalize a cumulative XLF; and
- interpret the results and connect simulated populations to observations.

**Classroom route.** The seven main exercises follow the same structure as the
2025 lab. Hints and solutions are collapsed—click them only when needed. The
metallicity and observational comparison is an optional extension.


## 1. Setting up the population run

### 1.1 Setting up POSYDON

The cells below import the necessary modules. We intentionally keep this setup
simple: the JupyterHub provides the official POSYDON 2.3 environment, so the
notebook does not enforce a version string, Git commit, or dataset metadata.


In [ ]:
from pathlib import Path
import contextlib
import json
import re
import shutil
import subprocess

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import posydon
from posydon.config import PATH_TO_POSYDON, PATH_TO_POSYDON_DATA
from posydon.popsyn.synthetic_population import Population, PopulationRunner
from posydon.utils import constants as const
from posydon.utils.common_functions import (
    orbital_separation_from_period,
    roche_lobe_radius,
)

mpl.rcParams["text.usetex"] = False
mpl.rcParams["font.family"] = "DejaVu Serif"
pd.set_option("display.max_columns", 18)

print("POSYDON version:", posydon.__version__)
print("POSYDON data:", PATH_TO_POSYDON_DATA)


### Define our working directory

All files created during the lab are kept in one directory on the school
JupyterHub. If you run the notebook elsewhere, change this path.


In [ ]:
WORK_DIR = Path("/home/jovyan/xrb_lab_test")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)


### 1.2 Loading additional XRB-related functions

The `xrb` module is not part of the installed POSYDON v2.3 release on the
JupyterHub yet. During testing, place `xrb.py` in the same directory as this
notebook. The import below automatically prefers the installed POSYDON module
after the code PR is merged, and otherwise uses the local file.


In [ ]:
import importlib.util

if importlib.util.find_spec("posydon.utils.xrb") is not None:
    # Final route after the POSYDON PR is merged.
    from posydon.utils.xrb import (
        accretion_luminosity,
        be_xray_luminosity,
        black_hole_radiative_efficiency,
        bondi_hoyle_accretion_rate,
        neutron_star_radiative_efficiency,
        wind_velocity,
    )
    XRB_SOURCE = "installed POSYDON package"
else:
    # Temporary route for testing with xrb.py beside this notebook.
    import xrb as local_xrb
    from xrb import (
        accretion_luminosity,
        be_xray_luminosity,
        black_hole_radiative_efficiency,
        bondi_hoyle_accretion_rate,
        neutron_star_radiative_efficiency,
        wind_velocity,
    )
    XRB_SOURCE = local_xrb.__file__

print("XRB utilities:", XRB_SOURCE)
print("Zero-spin BH efficiency:", black_hole_radiative_efficiency(0.0))


### 1.3 Configure and run a small snapshot population

First, copy the default population-parameter file into our working directory.
Keeping a local copy lets us see exactly which assumptions differ from the
POSYDON defaults.


In [ ]:
default_ini = (
    Path(PATH_TO_POSYDON) / "posydon" / "popsyn"
    / "population_params_default.ini"
)
population_ini = WORK_DIR / "population_params.ini"
shutil.copyfile(default_ini, population_ini)
print(population_ini)


As in the previous lab, we explore high-mass X-ray binaries in star-forming
galaxies. Observational star-formation rates are often interpreted assuming a
constant star-formation rate over the last 100 Myr. This is exactly what we
will assume here. Which population parameters need to change?

<details>
<summary>Click to reveal code</summary>

~~~python
star_formation = 'constant'
max_simulation_time = 1.0e8
~~~

</details>

We also want to run 100 binaries at two metallicities: solar and 10% solar.

<details>
<summary>Click to reveal code</summary>

~~~python
number_of_binaries = 100
metallicities = [1.0, 0.1]
~~~

</details>


In [ ]:
text = population_ini.read_text()
updates = {
    "number_of_binaries": "100",
    "metallicities": "[1.0, 0.1]",
    "star_formation": "'constant'",
    "max_simulation_time": "1.0e8",
}

for name, value in updates.items():
    text = re.sub(
        rf"(?m)^(\s*{name}\s*=\s*).*$",
        rf"\g<1>{value}",
        text,
        count=1,
    )

population_ini.write_text(text)
print("Updated:", population_ini)


Now it is time to run our test population. As before, we use the
`PopulationRunner` class to read the parameter file and evolve the population.
This may take a few minutes. The live population demonstrates the workflow,
but 100 binaries per metallicity are not enough for a scientifically smooth
XLF.


In [ ]:
RUN_SMALL_POPULATION = True

if RUN_SMALL_POPULATION:
    with contextlib.chdir(WORK_DIR):
        poprun = PopulationRunner(str(population_ini), verbose=True)
        poprun.evolve(overwrite=True)
else:
    print("Small population run skipped.")


Load the solar-metallicity population produced by the small run. We can inspect
the simulated mass, the number of systems, the detailed history of one binary,
history lengths, and formation channels.


In [ ]:
small_solar_candidates = list(
    WORK_DIR.glob("**/1e+00_Zsun_population.h5")
)

if small_solar_candidates:
    small_pop = Population(str(small_solar_candidates[0]))
    print(small_pop.mass_per_metallicity)
    print("Number of systems:", small_pop.number_of_systems)
    display(small_pop.history[5])
    print("History lengths:")
    display(small_pop.history_lengths)
    small_pop.calculate_formation_channels(mt_history=True)
    display(small_pop.formation_channels.head())
else:
    print("Run the previous cell to create the small population.")


Unfortunately, with a 100-binary population we cannot do much statistically.
To get a better feeling for what population synthesis can do, we use a
pre-calculated population with 100,000 binaries at solar metallicity. The 2026
school archive stores it under `populations/XRB_100K_pops/`.

During early JupyterHub testing, before the 2026 archive is installed, you may
point `PRECOMPUTED_DIR` to an equivalent test-data directory instead.


In [ ]:
PRECOMPUTED_DIR = (
    Path(PATH_TO_POSYDON_DATA).parent
    / "2026_school_data" / "populations" / "XRB_100K_pops"
)

# Temporary testing alternative, if needed:
# PRECOMPUTED_DIR = (
#     Path(PATH_TO_POSYDON_DATA).parent
#     / "2025_school_data" / "populations" / "XRB_100K_pops"
# )

solar_population_path = PRECOMPUTED_DIR / "1e+00_Zsun_population.h5"
low_z_population_path = PRECOMPUTED_DIR / "1e-01_Zsun_population.h5"

pop = Population(str(solar_population_path))
print("Loaded:", solar_population_path)


<div class='alert alert-success'>

### Exercise 1: Inspect the population

Check the total number of binaries in this population, the simulated mass, how
many unique formation pathways the binaries followed, and how many binaries
went through each channel.

In this lab we mainly care about the binary properties at 100 Myr. Therefore,
we usually inspect `oneline`, which contains initial and final properties,
rather than the complete `history`. Print a list of all the `oneline` keys.

</div>


In [ ]:
# Write your code for Exercise 1 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

~~~python
print("The simulated mass and number of binaries per metallicity are:")
print(pop.mass_per_metallicity)
print("Number of systems:", pop.number_of_systems)

pop.calculate_formation_channels(mt_history=True)
print("Number of systems following each formation channel:")
print(pop.formation_channels.value_counts())

print("The oneline keys are:")
print(", ".join(list(pop.oneline[0].keys())))
~~~

</details>
</div>


## 2. Selecting the XRB population

Only a small fraction of the full population are potential XRBs, so we do not
need to carry every system through the analysis. We follow the logic of the
POSYDON transient-population tutorial, but adapt it because XRBs are phases
rather than instantaneous events.


<div class='alert alert-success'>

### Exercise 2: Select candidate XRBs

Filter the population to select potential XRBs. One component must be a
neutron star or black hole, while the other must be a normal, non-compact star.
Remove systems disrupted during a supernova, failed systems, and initially
overflowing systems. Save the indices into a list named `selected_indices`.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

Besides `NS` and `BH`, exclude `WD` and `massless_remnant` from the possible
donor states. Create a temporary table from `oneline` containing
`S1_state_f`, `S2_state_f`, and `state_f`, then build a Boolean mask.

The condition should allow either S1 to be the compact object and S2 the
donor, or S2 to be the compact object and S1 the donor. It should not select a
double compact object.

</details>
</div>


In [ ]:
# Write your code for Exercise 2 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

~~~python
tmp_data = pop.oneline.select(
    columns=["state_f", "S1_state_f", "S2_state_f"]
)

compact = {"BH", "NS"}
exclude_as_donor = {"BH", "NS", "WD", "massless_remnant"}
bad_binary_states = {"disrupted", "ERR", "initial_RLOF"}

s1_compact = tmp_data["S1_state_f"].isin(compact)
s2_compact = tmp_data["S2_state_f"].isin(compact)
s1_normal = ~tmp_data["S1_state_f"].isin(exclude_as_donor)
s2_normal = ~tmp_data["S2_state_f"].isin(exclude_as_donor)
bound_and_valid = ~tmp_data["state_f"].isin(bad_binary_states)

mask = (
    ((s1_compact & s2_normal) | (s2_compact & s1_normal))
    & bound_and_valid
)
selected_indices = tmp_data.index[mask].to_list()

print(f"Selected {len(selected_indices):,} candidate XRBs")
display(pop.oneline[selected_indices].head())
~~~

</details>
</div>


In [ ]:
tmp_data = pop.oneline.select(
    columns=["state_f", "S1_state_f", "S2_state_f"]
)

compact = {"BH", "NS"}
exclude_as_donor = {"BH", "NS", "WD", "massless_remnant"}
bad_binary_states = {"disrupted", "ERR", "initial_RLOF"}

s1_compact = tmp_data["S1_state_f"].isin(compact)
s2_compact = tmp_data["S2_state_f"].isin(compact)
s1_normal = ~tmp_data["S1_state_f"].isin(exclude_as_donor)
s2_normal = ~tmp_data["S2_state_f"].isin(exclude_as_donor)
bound_and_valid = ~tmp_data["state_f"].isin(bad_binary_states)

mask = (
    ((s1_compact & s2_normal) | (s2_compact & s1_normal))
    & bound_and_valid
)
selected_indices = tmp_data.index[mask].to_list()

print(f"Selected {len(selected_indices):,} candidate XRBs")
display(pop.oneline[selected_indices].head())


Before continuing, save this filtered population into a new file and reload it
as `XRB_pop`. The simulated mass remains the mass of the original underlying
population; filtering does not redefine the population normalization.


In [ ]:
xrb_selection_path = WORK_DIR / "XRBs.h5"
pop.export_selection(
    selected_indices,
    str(xrb_selection_path),
    append=False,
    overwrite=True,
)

XRB_pop = Population(str(xrb_selection_path))
print("Candidate systems:", XRB_pop.number_of_systems)
print("Underlying simulated mass:", XRB_pop.mass_per_metallicity)


The next step is to construct a pandas DataFrame containing only the
information needed for the XLF. For the later calculations it is convenient
to replace POSYDON's S1/S2 notation with donor/accretor notation. We identify
the BH or NS as the accretor and the normal star as the donor.

Unlike a transient-event selection, this snapshot calculation only needs the
final state in `oneline`; the function keeps the `history_chunk` argument so it
can use POSYDON's standard population-selection interface.


In [ ]:
def XRB_selection_function(history_chunk, oneline_chunk,
                           formation_channels_chunk=None):
    '''Create a snapshot table with basic donor/accretor properties.'''
    indices = oneline_chunk.index.to_numpy()
    df_XRBs = pd.DataFrame(index=indices)

    df_XRBs["time"] = oneline_chunk["time_f"] * 1.0e-6  # Myr
    df_XRBs["metallicity"] = oneline_chunk["metallicity"]

    compact_types = {"BH", "NS"}
    donor_mass = []
    accretor_mass = []
    donor_state = []
    accretor_state = []

    for s1, s2, m1, m2 in zip(
        oneline_chunk["S1_state_f"],
        oneline_chunk["S2_state_f"],
        oneline_chunk["S1_mass_f"],
        oneline_chunk["S2_mass_f"],
    ):
        if s1 in compact_types and s2 not in compact_types:
            accretor_mass.append(m1)
            accretor_state.append(s1)
            donor_mass.append(m2)
            donor_state.append(s2)
        elif s2 in compact_types and s1 not in compact_types:
            accretor_mass.append(m2)
            accretor_state.append(s2)
            donor_mass.append(m1)
            donor_state.append(s1)
        else:
            accretor_mass.append(np.nan)
            accretor_state.append(None)
            donor_mass.append(np.nan)
            donor_state.append(None)

    df_XRBs["donor_mass"] = donor_mass
    df_XRBs["accretor_mass"] = accretor_mass
    df_XRBs["donor_state"] = donor_state
    df_XRBs["accretor_state"] = accretor_state
    return df_XRBs


Test the first version of the function on one binary. Reading the result
horizontally makes it easy to confirm that the compact object is always the
accretor, regardless of whether it was S1 or S2.


In [ ]:
display(
    XRB_selection_function(
        XRB_pop.history[0], XRB_pop.oneline[0], None
    ).T
)


<div class='alert alert-success'>

### Exercise 3: Store the properties needed for accretion

Extend `XRB_selection_function` to include donor and accretor radii,
mass-loss/mass-transfer rates, orbital period and eccentricity, the donor's
surface rotation relative to critical, the donor luminosity and surface
hydrogen fraction, the donor helium-core mass, and the accretor spin.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

POSYDON stores many quantities as base-10 logarithms. In particular, radii,
luminosities, wind mass-loss rates, and RLO mass-transfer rates must be
converted before they are passed to `xrb`.

Useful fields include `S1_log_R_f`, `S2_log_R_f`, `S1_log_L_f`,
`S2_log_L_f`, `S1_lg_wind_mdot_f`, `S2_lg_wind_mdot_f`,
`lg_mtransfer_rate_f`, `S1_surface_h1_f`, `S2_surface_h1_f`, and the
corresponding spin, rotation, and helium-core-mass columns.

</details>
</div>


In [ ]:
# Write your additions for Exercise 3 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

The complete reference implementation is given below, where these quantities are immediately connected to the luminosity calculation.

~~~python
# Inside the donor/accretor mapping, use the same S1/S2 choice for every
# stellar property. Convert logarithmic quantities only after the mapping.

out["donor_radius"] = 10.0**choose(
    "S1_log_R_f", "S2_log_R_f", accretor=False
).astype(float)
out["donor_luminosity"] = 10.0**choose(
    "S1_log_L_f", "S2_log_L_f", accretor=False
).astype(float)
out["donor_wind_mass_loss"] = 10.0**choose(
    "S1_lg_wind_mdot_f", "S2_lg_wind_mdot_f", accretor=False
).astype(float)
out["rlo_mass_transfer_rate"] = 10.0**df[
    "lg_mtransfer_rate_f"
].to_numpy(float)
~~~

</details>
</div>


### 2.1 From mass supply to X-ray luminosity

We need one more quantity, but it is the central observable of this lab: the
X-ray luminosity. We follow the classical approach described in Section 2.2
of [Misra et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023A%26A...672A..99M/abstract).

In the 2025 exercise we wrote approximate luminosity functions directly in
the notebook and, for simplicity, neglected Be-XRBs and geometrical beaming.
This year those reusable calculations live in `xrb.py`. Moving them into the
POSYDON code base lets the equations be tested once and shared by other
analyses, while selection and population-level assumptions remain visible in
the notebook.

For radiative efficiency $\eta$, a sub-Eddington accretion flow has

$$
L_{\mathrm{bol}} = \eta\,\dot{M}\,c^2.
$$

For black holes, `black_hole_radiative_efficiency` calculates the
spin-dependent Novikov–Thorne efficiency. For neutron stars we use the
Newtonian surface-accretion estimate

$$
\eta_{\mathrm{NS}} = \frac{G M_{\mathrm{NS}}}
{R_{\mathrm{NS}}c^2}.
$$

The Eddington luminosity and corresponding mass-accretion rate are

$$
L_{\mathrm{Edd}} =
\frac{4\pi G M c}{0.2\left(1+X_{\mathrm{surf}}\right)},
\qquad
\dot{M}_{\mathrm{Edd}} =
\frac{L_{\mathrm{Edd}}}{\eta c^2}.
$$

Here $X_{\mathrm{surf}}$ is the donor's surface hydrogen mass fraction. Define
$\dot{m}=\dot{M}/\dot{M}_{\mathrm{Edd}}$. Above the Eddington rate, the
classical logarithmic enhancement is

$$
L_{\mathrm{iso}} =
\frac{L_{\mathrm{Edd}}}{b}\left(1+\ln\dot{m}\right).
$$

The King beaming factor is $b=1$ through $\dot{m}=8.5$ and
$b=73/\dot{m}^{2}$ above that transition, with a floor
$b_{\min}=3.2\times10^{-3}$. The module returns a bolometric,
isotropic-equivalent luminosity. For RLO and wind-fed systems, the notebook
explicitly adopts the approximate band correction

$$
L_{0.5-8\,\mathrm{keV}} = 0.5\,L_{\mathrm{bol}}.
$$

This factor is an observational assumption, not part of the reusable physics.
Magnetic neutron-star corrections and advanced accretion-flow prescriptions
are outside the scope of this lab.


### 2.2 RLO, wind-fed, and Be-XRB channels

- **Roche-lobe overflow (RLO):** the donor fills its Roche lobe and matter
  flows through the inner Lagrange point. We use POSYDON's mass-transfer rate
  as the supplied accretion rate.
- **Wind-fed XRB:** a detached compact object captures a fraction of the
  donor's wind. `wind_velocity` estimates the terminal speed, and
  `bondi_hoyle_accretion_rate` evaluates the deterministic orbit-averaged
  Bondi–Hoyle–Lyttleton capture rate.
- **Be-XRB:** a rapidly rotating H-rich B star can supply a decretion disc.
  The notebook identifies plausible systems, while `be_xray_luminosity`
  provides the empirical orbital-period–luminosity relation.

The wind prescription needs the donor's helium-core mass as an evolutionary
proxy. This is different from the helium-core radius: a mass and a radius have
different dimensions and cannot be substituted for one another.

In this tutorial, a detached candidate is labeled Be when the donor is an
H-rich core-H-burning star with $M\geq6\,M_\odot$, surface rotation at least
70% of critical, $10\leq P_{\mathrm{orb}}\leq300$ days, and a notional
decretion disc extending to 100 stellar radii that reaches the donor's
periastron Roche lobe. We assign a 10% duty cycle later. This is a model choice
and a major uncertainty—not a definition of every observed Be-XRB.


In [ ]:
def XRB_selection_function(history_chunk, oneline_chunk,
                           formation_channels_chunk=None):
    '''Create the snapshot XRB table and calculate classical luminosities.'''
    df = oneline_chunk.copy()
    compact = {"BH", "NS"}
    excluded = {"BH", "NS", "WD", "massless_remnant"}
    bad_states = {"disrupted", "ERR", "initial_RLOF"}

    s1_compact = df["S1_state_f"].isin(compact)
    s2_compact = df["S2_state_f"].isin(compact)
    keep = (
        (
            (s1_compact & ~df["S2_state_f"].isin(excluded))
            | (s2_compact & ~df["S1_state_f"].isin(excluded))
        )
        & ~df["state_f"].isin(bad_states)
    )
    df = df.loc[keep].copy()
    s1_compact = df["S1_state_f"].isin(compact).to_numpy()

    def choose(s1_column, s2_column, accretor=True):
        use_s1 = s1_compact if accretor else ~s1_compact
        return np.where(use_s1, df[s1_column], df[s2_column])

    out = pd.DataFrame(index=df.index)
    out["time"] = df["time_f"].to_numpy(float) * 1.0e-6  # Myr
    out["time_of_birth"] = df["time_i"].to_numpy(float) * 1.0e-6
    out["metallicity"] = df["metallicity"].to_numpy(float)
    out["binary_state"] = df["state_f"].to_numpy()
    out["orbital_period"] = df["orbital_period_f"].to_numpy(float)
    out["eccentricity"] = df["eccentricity_f"].to_numpy(float)

    out["accretor_state"] = choose("S1_state_f", "S2_state_f")
    out["accretor_mass"] = choose(
        "S1_mass_f", "S2_mass_f"
    ).astype(float)
    out["accretor_radius"] = 10.0**choose(
        "S1_log_R_f", "S2_log_R_f"
    ).astype(float)
    out["accretor_spin"] = choose(
        "S1_spin_f", "S2_spin_f"
    ).astype(float)

    out["donor_state"] = choose(
        "S1_state_f", "S2_state_f", accretor=False
    )
    out["donor_mass"] = choose(
        "S1_mass_f", "S2_mass_f", accretor=False
    ).astype(float)
    out["donor_radius"] = 10.0**choose(
        "S1_log_R_f", "S2_log_R_f", accretor=False
    ).astype(float)
    out["donor_luminosity"] = 10.0**choose(
        "S1_log_L_f", "S2_log_L_f", accretor=False
    ).astype(float)
    out["donor_surface_h1"] = choose(
        "S1_surface_h1_f", "S2_surface_h1_f", accretor=False
    ).astype(float)

    donor_he_core_mass = choose(
        "S1_he_core_mass_f", "S2_he_core_mass_f", accretor=False
    ).astype(float)
    out["donor_he_core_mass"] = np.where(
        np.isfinite(donor_he_core_mass) & (donor_he_core_mass >= 0.0),
        donor_he_core_mass,
        0.0,
    )
    out["donor_rotation_fraction"] = choose(
        "S1_surf_avg_omega_div_omega_crit_f",
        "S2_surf_avg_omega_div_omega_crit_f",
        accretor=False,
    ).astype(float)

    donor_log_wind = choose(
        "S1_lg_wind_mdot_f",
        "S2_lg_wind_mdot_f",
        accretor=False,
    ).astype(float)
    out["donor_wind_mass_loss"] = 10.0**donor_log_wind
    out["rlo_mass_transfer_rate"] = 10.0**df[
        "lg_mtransfer_rate_f"
    ].to_numpy(float)

    out["orbital_separation"] = orbital_separation_from_period(
        out["orbital_period"],
        out["accretor_mass"],
        out["donor_mass"],
    )
    donor_roche_lobe_periastron = roche_lobe_radius(
        out["donor_mass"],
        out["accretor_mass"],
        out["orbital_separation"] * (1.0 - out["eccentricity"]),
    )

    detached = out["binary_state"].eq("detached")
    rlo = out["binary_state"].isin(["RLO1", "RLO2"])
    be = (
        detached
        & out["donor_state"].eq("H-rich_Core_H_burning")
        & (out["donor_mass"] >= 6.0)
        & (out["donor_rotation_fraction"] >= 0.7)
        & out["orbital_period"].between(10.0, 300.0)
        & (donor_roche_lobe_periastron
           <= 100.0 * out["donor_radius"])
    )
    wind = detached & ~be
    out["accretion_mode"] = np.select(
        [be, rlo, wind],
        ["Be", "RLO", "wind"],
        default="other",
    )

    bh = out["accretor_state"].eq("BH").to_numpy()
    bh_spin = out["accretor_spin"].to_numpy(float)
    bh_spin = np.where(np.isfinite(bh_spin), bh_spin, 0.0)

    ns_radius = out["accretor_radius"].to_numpy(float)
    ns_radius = np.where(
        np.isfinite(ns_radius) & (ns_radius > 0.0),
        ns_radius,
        1.25e6 / const.Rsun,
    )

    efficiency = np.where(
        bh,
        black_hole_radiative_efficiency(bh_spin),
        neutron_star_radiative_efficiency(
            out["accretor_mass"], ns_radius
        ),
    )
    out["radiative_efficiency"] = efficiency

    speed = wind_velocity(
        out["donor_mass"],
        out["donor_radius"],
        out["donor_luminosity"],
        out["donor_wind_mass_loss"],
        out["donor_surface_h1"],
        out["donor_he_core_mass"],
        scheme="Kudritzki+2000",
    )
    wind_rate = bondi_hoyle_accretion_rate(
        out["accretor_mass"],
        out["donor_mass"],
        out["donor_wind_mass_loss"],
        out["orbital_separation"],
        out["eccentricity"],
        speed,
    )
    out["wind_accretion_rate"] = wind_rate

    supplied_rate = np.where(
        rlo,
        out["rlo_mass_transfer_rate"],
        wind_rate,
    )
    luminosity, beaming, Eddington_ratio = accretion_luminosity(
        supplied_rate,
        out["accretor_mass"],
        out["donor_surface_h1"],
        efficiency,
    )

    classical_lx = 0.5 * luminosity
    out["L_bol_iso"] = luminosity
    out["beaming_factor"] = beaming
    out["Eddington_ratio"] = Eddington_ratio
    out["Lx_rlo"] = np.where(rlo, classical_lx, 0.0)
    out["Lx_bhl"] = np.where(wind, classical_lx, 0.0)
    out["Lx_be"] = 0.0
    out.loc[be, "Lx_be"] = be_xray_luminosity(
        out.loc[be, "orbital_period"]
    )
    out.loc[be, "beaming_factor"] = 1.0
    out["Lx_tot"] = out[["Lx_rlo", "Lx_bhl", "Lx_be"]].sum(axis=1)
    out["duty_cycle"] = np.where(be, 0.1, 1.0)

    if formation_channels_chunk is not None:
        out["formation_channel"] = formation_channels_chunk.loc[
            out.index, "channel"
        ]

    return out.loc[out["accretion_mode"] != "other"]


<div class='alert alert-success'>

### Exercise 4: Apply the luminosity selection function

Try the new `XRB_selection_function`, including its X-ray luminosities, first
on one binary and then on the entire candidate population. Inspect the output
columns and count the systems in each accretion channel.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

Use `XRB_pop.history[index]` and `XRB_pop.oneline[index]` for one system. For
the whole population, pass the selection function to
`XRB_pop.create_transient_population` and give the result a descriptive name.

</details>
</div>


In [ ]:
# Write your code for Exercise 4 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

~~~python
example = XRB_selection_function(
    XRB_pop.history[10], XRB_pop.oneline[10], None
)
display(example.T)

XRBs = XRB_pop.create_transient_population(
    XRB_selection_function,
    "XRB_snapshot",
)
display(XRBs.population.head())
print(XRBs.population["accretion_mode"].value_counts())
~~~

</details>
</div>


In [ ]:
example = XRB_selection_function(
    XRB_pop.history[10], XRB_pop.oneline[10], None
)
display(example.T)

XRBs = XRB_pop.create_transient_population(
    XRB_selection_function,
    "XRB_snapshot",
)
display(XRBs.population.head())
print(XRBs.population["accretion_mode"].value_counts())


## 3. X-ray luminosity function of extragalactic X-ray binaries

The **X-ray luminosity function (XLF)** of an extragalactic XRB population
describes how many XRBs exist in a galaxy as a function of their X-ray
luminosity. It can be expressed in two common forms.

- **Differential XLF**

  $$
  \Phi(L_X) = \frac{dN}{dL_X}
  \qquad \mathrm{or} \qquad
  \frac{dN}{d\log L_X}.
  $$

  This gives the number of sources per unit luminosity, or per unit
  logarithmic luminosity interval.

- **Cumulative XLF**

  $$
  N(>L_X) = \int_{L_X}^{\infty}\Phi(L)\,dL.
  $$

  This gives the total number of XRBs brighter than $L_X$.

### Observational usage

Cumulative XLFs are often shown because they avoid the statistical noise
introduced by binning sparse data, power-law distributions are easy to inspect
in log–log space, and galaxies with different numbers of sources can be
compared after applying a physical normalization.

### Characteristic forms

- **High-mass XRBs** in star-forming galaxies have an approximately power-law
  XLF whose normalization scales with star-formation rate. The detailed shape,
  including breaks and the bright end, contains information about binary
  evolution and accretion physics.
- **Low-mass XRBs** in older populations show characteristic breaks around
  $L_X\sim10^{37}$–$10^{38}\,\mathrm{erg\,s^{-1}}$, and their normalization
  primarily scales with the stellar mass of the host galaxy.

We use the cumulative XLF, the standard diagnostic in many extragalactic
studies. Remember that our snapshot contains a Monte Carlo sample: a raw count
is not yet a prediction for a physical galaxy.


<div class='alert alert-success'>

### Exercise 5: Plot the unnormalized XLF

Create a figure showing the cumulative XLF of the selected XRB population.
Use the total 0.5–8 keV luminosity `Lx_tot` for every XRB.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

Convert `XRBs.population["Lx_tot"]` to a one-dimensional NumPy array. Remove
non-finite values and impose a minimum luminosity.

Sort the luminosities. For each sorted luminosity, the cumulative ordinate is
the number of systems at least that bright. One reliable implementation is to
reverse the sorted array of unit weights, cumulatively sum it, and reverse the
result back.

</details>
</div>


In [ ]:
# Write your code for Exercise 5 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

~~~python
L = pd.to_numeric(XRBs.population["Lx_tot"]).to_numpy(dtype=float)
valid = np.isfinite(L) & (L > 1.0e30)
L_valid = L[valid]

order = np.argsort(L_valid)
L_sorted = L_valid[order]
N_gt = np.cumsum(np.ones_like(L_sorted)[::-1])[::-1]

fig, ax = plt.subplots(figsize=(5, 5))
ax.step(L_sorted, N_gt, where="post")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("X-ray luminosity (erg/s)")
ax.set_ylabel("N(>Lx)")
ax.set_xlim(1.0e35, 1.0e41)
ax.grid(alpha=0.2, which="both")
plt.show()
~~~

</details>
</div>


### 3.1 A proper normalization of the simulated XLF

The vertical axis of the raw XLF is arbitrary. Had we simulated ten times more
binaries, we would expect roughly ten times more XRBs. To compare with
observations, we normalize the XLF to something physical. For star-forming
galaxies, the usual normalization is per unit star-formation rate,
$M_\odot\,\mathrm{yr}^{-1}$.

POSYDON can calculate the probability of each modeled system per unit stellar
mass formed. This machinery can reweight a population to initial binary
distributions different from those used to sample it; here we use it in its
simplest form.

We simulated only systems with primary masses above the grid limit because
low-mass binaries do not form the HMXBs studied here. For a probability per
unit *total* stellar mass, however, the normalization of the IMF must extend to
low masses. We therefore set the primary-mass minimum to
$0.1\,M_\odot$ and use the full mass-ratio interval.


In [ ]:
population_parameters = XRBs.ini_params.copy()
population_parameters["q_min"] = 0.0
population_parameters["q_max"] = 1.0
population_parameters["primary_mass_min"] = 0.1

weights = XRBs.calculate_model_weights(
    model_weights_identifier="base_IMF",
    population_parameters=population_parameters,
)

if isinstance(weights, pd.DataFrame):
    weights = weights.iloc[:, 0]

weights = np.asarray(weights, dtype=float)
print("Number of model weights:", len(weights))


<div class='alert alert-success'>

### Exercise 6: Normalize the XLF by star-formation rate

Normalize the XLF per unit star-formation rate. Assume a constant
$1\,M_\odot\,\mathrm{yr}^{-1}$ star-formation rate over the last 100 Myr.

In addition to the population weight, account for the probability of seeing a
beamed source and the Be-XRB duty cycle.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

A constant $1\,M_\odot\,\mathrm{yr}^{-1}$ over $10^8$ yr forms
$10^8\,M_\odot$ of stars. Multiply the model weights by this mass.

In Exercise 5 the cumulative sum used unit weights. Replace those ones with
the physical weights, sorted in exactly the same order as the luminosities.
The observability factor is `beaming_factor * duty_cycle`.

</details>
</div>


In [ ]:
# Write your code for Exercise 6 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

~~~python
SFR = 1.0             # Msun / yr
TIME_WINDOW = 1.0e8   # yr
TOTAL_FORMED_MASS = SFR * TIME_WINDOW

observability = (
    XRBs.population["beaming_factor"].to_numpy(float)
    * XRBs.population["duty_cycle"].to_numpy(float)
)
physical_weights = weights * TOTAL_FORMED_MASS * observability

def cumulative_xlf(luminosity, source_weights, minimum=1.0e35):
    luminosity = np.asarray(luminosity, dtype=float)
    source_weights = np.asarray(source_weights, dtype=float)
    valid = (
        np.isfinite(luminosity)
        & np.isfinite(source_weights)
        & (luminosity >= minimum)
        & (source_weights >= 0.0)
    )
    luminosity = luminosity[valid]
    source_weights = source_weights[valid]
    order = np.argsort(luminosity)
    x = luminosity[order]
    y = np.cumsum(source_weights[order][::-1])[::-1]
    return x, y

luminosity = XRBs.population["Lx_tot"].to_numpy(float)
x, y = cumulative_xlf(luminosity, physical_weights)

fig, ax = plt.subplots(figsize=(5, 5))
ax.step(x, y, where="post", color="black")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("X-ray luminosity (erg/s)")
ax.set_ylabel("N(>Lx) / SFR")
ax.set_xlim(1.0e35, 1.0e41)
ax.grid(alpha=0.2, which="both")
plt.show()
~~~

</details>
</div>


In [ ]:
SFR = 1.0             # Msun / yr
TIME_WINDOW = 1.0e8   # yr
TOTAL_FORMED_MASS = SFR * TIME_WINDOW

observability = (
    XRBs.population["beaming_factor"].to_numpy(float)
    * XRBs.population["duty_cycle"].to_numpy(float)
)
physical_weights = weights * TOTAL_FORMED_MASS * observability

def cumulative_xlf(luminosity, source_weights, minimum=1.0e35):
    luminosity = np.asarray(luminosity, dtype=float)
    source_weights = np.asarray(source_weights, dtype=float)
    valid = (
        np.isfinite(luminosity)
        & np.isfinite(source_weights)
        & (luminosity >= minimum)
        & (source_weights >= 0.0)
    )
    luminosity = luminosity[valid]
    source_weights = source_weights[valid]
    order = np.argsort(luminosity)
    x = luminosity[order]
    y = np.cumsum(source_weights[order][::-1])[::-1]
    return x, y

luminosity = XRBs.population["Lx_tot"].to_numpy(float)
x, y = cumulative_xlf(luminosity, physical_weights)

fig, ax = plt.subplots(figsize=(5, 5))
ax.step(x, y, where="post", color="black")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("X-ray luminosity (erg/s)")
ax.set_ylabel("N(>Lx) / SFR")
ax.set_xlim(1.0e35, 1.0e41)
ax.grid(alpha=0.2, which="both")
plt.show()


<div class='alert alert-success'>

### Exercise 7: Split the physical XRB subpopulations

Explore the XLF by splitting it according to accretor type (NS or BH) and
accretion channel (RLO, wind-fed, or Be-XRB). Which populations dominate the
bright and faint ends? Return to the efficiency prediction you made from the
equations above.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

Build masks from the `accretor_state` and `accretion_mode` columns. Apply each
mask to both the luminosity array and the physical-weight array before calling
`cumulative_xlf`. Skip empty subpopulations.

</details>
</div>


In [ ]:
# Write your code for Exercise 7 here.


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution (click to reveal):</summary></b>

~~~python
fig, ax = plt.subplots(figsize=(7, 5.2))
styles = {
    ("BH", "RLO"): ("tab:blue", "-"),
    ("NS", "RLO"): ("tab:orange", "-"),
    ("BH", "wind"): ("tab:blue", "--"),
    ("NS", "wind"): ("tab:orange", "--"),
    ("BH", "Be"): ("tab:green", ":"),
    ("NS", "Be"): ("tab:red", ":"),
}

table = XRBs.population
for (compact_type, mode), (color, linestyle) in styles.items():
    mask = (
        table["accretor_state"].eq(compact_type)
        & table["accretion_mode"].eq(mode)
    ).to_numpy()
    x_sub, y_sub = cumulative_xlf(
        luminosity[mask], physical_weights[mask]
    )
    if len(x_sub):
        ax.step(
            x_sub, y_sub, where="post", color=color,
            linestyle=linestyle, label=f"{compact_type} + {mode}"
        )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e35, 1.0e41)
ax.set_xlabel("X-ray luminosity (erg/s)")
ax.set_ylabel("N(>Lx) / SFR")
ax.grid(alpha=0.2, which="both")
ax.legend(ncol=2, fontsize=9)
plt.show()
~~~

</details>
</div>


### Interpreting the split XLF

- Neutron stars can have a higher surface-accretion efficiency than a
  non-spinning black hole, but their lower masses also give them lower
  Eddington luminosities.
- RLO can sustain much larger supplied rates than wind capture and often
  controls the bright end.
- King beaming increases the on-axis isotropic-equivalent luminosity while
  reducing the probability that a randomly oriented observer lies inside the
  beam.
- The predicted Be-XRB population depends strongly on the selection criteria
  and adopted duty cycle.

The XLF tests a combined model: binary evolution, snapshot sampling,
accretion prescriptions, band correction, beaming orientation, duty cycles,
and IMF normalization. Agreement or disagreement cannot automatically be
assigned to only one ingredient.


<div class='alert alert-success'>

### (Optional) Exercise 8: Compare metallicity and observations

Compare the simulated XLF with an observed one, for example Figure 3 of
[Lehmer et al. (2021)](https://arxiv.org/pdf/2011.09476). Do you see
similarities or obvious discrepancies?

Then load the available population at 10% solar metallicity and repeat the
analysis. Does the same metallicity trend appear? Speculate about why XLF
shape and normalization may depend on metallicity. Consider wind mass loss,
compact-object masses, binary interaction, and donor structure.

</div>


<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint (click to reveal):</summary></b>

Repeat the selection, luminosity, weighting, and cumulative-XLF steps with
`low_z_population_path`. Before overlaying observations, check the energy
band, absorption convention, differential versus cumulative definition, SFR
calibration and IMF, completeness limit, and treatment of unresolved or
background sources.

</details>
</div>


In [ ]:
# Optional scaffold for the 0.1 Zsun population.
low_z_pop = Population(str(low_z_population_path))
print(low_z_pop.number_of_systems)
print(low_z_pop.mass_per_metallicity)


## 4. Assumptions, limitations, and takeaways

### Assumptions made in this lab

- A 100 Myr snapshot of a constant-SFR population.
- Classical RLO and deterministic orbit-averaged wind capture.
- Novikov–Thorne BH and Newtonian NS efficiencies.
- King super-Eddington beaming with a fixed minimum beaming factor.
- A factor 0.5 bolometric-to-0.5–8 keV conversion for RLO/wind systems.
- An empirical Be period–luminosity relation and 10% duty cycle.
- No GRRMHD, magnetic NS, wind-disc-formation, absorption, or detector model.

### Common pitfalls

- POSYDON `lg_*` columns are base-10 logarithms. Convert them before passing
  rates or radii to the public XRB functions.
- Keep the luminosity array and its weights aligned when sorting or masking.
- An isotropic-equivalent luminosity and a beaming probability are different
  quantities; apply both consistently.
- A smooth-looking XLF can still be statistically unconverged. Research
  calculations should compare multiple population sizes or random seeds.

### Takeaways

1. XRBs are evolving phases, so the population-sampling method is part of the
   prediction.
2. POSYDON evolves the binaries; the `xrb` module connects accretion
   properties to classical X-ray luminosities.
3. Selection, band corrections, duty cycles, and population normalization are
   analysis choices and remain explicit in the notebook.
4. Splitting the XLF by compact-object type and accretion channel is essential
   for diagnosing the underlying physics.

### Further reading

- Misra et al. (2023), *A&A* 672, A99 — detailed HMXB XLF modeling.
- King (2008), *MNRAS* 385, L113 — super-Eddington beaming.
- Hurley, Tout & Pols (2002), *MNRAS* 329, 897 — binary-evolution wind capture.
- Dai, Liu & Li (2006), *ApJ* 653, 1410 — Be-XRB period relation.
- Lehmer et al. (2021), *ApJ* 907, 17 — observed metallicity-dependent XLFs.
